In [3]:
import pandas as pd
import plotly.express as px

# Importing the data

In [18]:
doe = pd.read_csv("2019-20_Demographic_Snapshot_-_Borough_20251117.csv")
sca = pd.read_csv("New_Capacity_Program_By_Borough_20251117.csv")

print("DOE original columns:", doe.columns.tolist())
print("SCA original columns:", sca.columns.tolist())

DOE original columns: ['Borough', 'Year', 'Total Enrollment', 'Grade 3K+PK (Half Day & Full Day)', 'Grade K', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4', 'Grade 5', 'Grade 6', 'Grade 7', 'Grade 8', 'Grade 9', 'Grade 10', 'Grade 11', 'Grade 12', '# Female', '% Female', '# Male', '% Male', '# Asian', '% Asian', '# Black', '% Black', '# Hispanic', '% Hispanic', '# Multiple Race Categories Not Represented', '% Multiple Race Categories Not Represented', '# White', '% White', '# Students with Disabilities', '% Students with Disabilities', '# English Language Learners', '% English Language Learners', '# Poverty', '% Poverty', 'Economic Need Index']
SCA original columns: ['DISTRICT', 'BOROUGH', 'SMALL PS # BLDGS', 'SMALL PS # SEATS', 'SMALL PS COST', 'PS/IS # BLDGS', 'PS/IS # SEATS', 'PS/IS COST', 'IS/HS # BLDGS', 'IS/HS # SEATS', 'IS/HS COST']


In [19]:
doe.info()
sca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 38 columns):
 #   Column                                      Non-Null Count  Dtype 
---  ------                                      --------------  ----- 
 0   Borough                                     25 non-null     object
 1   Year                                        25 non-null     object
 2   Total Enrollment                            25 non-null     object
 3   Grade 3K+PK (Half Day & Full Day)           25 non-null     object
 4   Grade K                                     25 non-null     object
 5   Grade 1                                     25 non-null     object
 6   Grade 2                                     25 non-null     object
 7   Grade 3                                     25 non-null     object
 8   Grade 4                                     25 non-null     object
 9   Grade 5                                     25 non-null     object
 10  Grade 6                     

In [21]:
doe.columns = (
    doe.columns
       .str.strip()
       .str.lower()
       .str.replace(" ", "_")
)
print("DOE cleaned columns:", doe.columns.tolist())

DOE cleaned columns: ['borough', 'year', 'total_enrollment', 'grade_3k+pk_(half_day_&_full_day)', 'grade_k', 'grade_1', 'grade_2', 'grade_3', 'grade_4', 'grade_5', 'grade_6', 'grade_7', 'grade_8', 'grade_9', 'grade_10', 'grade_11', 'grade_12', '#_female', '%_female', '#_male', '%_male', '#_asian', '%_asian', '#_black', '%_black', '#_hispanic', '%_hispanic', '#_multiple_race_categories_not_represented', '%_multiple_race_categories_not_represented', '#_white', '%_white', '#_students_with_disabilities', '%_students_with_disabilities', '#_english_language_learners', '%_english_language_learners', '#_poverty', '%_poverty', 'economic_need_index']


# Clean Demographics Snapshot Dataset

In [23]:
doe["borough"] = doe["borough"].astype(str).str.title().str.strip()
doe["total_enrollment"] = (
    doe["total_enrollment"]
      .astype(str)
      .str.replace(",", "", regex=False)
      .astype(float)
)

In [16]:
sca.columns = (sca.columns
                 .str.strip()
                 .str.lower()
                 .str.replace(" ", "_")
                 .str.replace("/", "_")
                 .str.replace("#", "")
)

# Check what they look like now
print("Clean SCA columns:", sca.columns.tolist())

Clean SCA columns: ['district', 'borough', 'small_ps__bldgs', 'small_ps__seats', 'small_ps_cost', 'ps_is__bldgs', 'ps_is__seats', 'ps_is_cost', 'is_hs__bldgs', 'is_hs__seats', 'is_hs_cost']


In [25]:
# keep year as string ("2019-20")
doe["year"] = doe["year"].astype(str)

# latest year in this file
latest_year = doe["year"].max()
print("Latest DOE year:", latest_year)

# filter to latest year and aggregate enrollment by borough
doe_latest = doe[doe["year"] == latest_year]

doe_boro = (
    doe_latest
      .groupby("borough", as_index=False)["total_enrollment"]
      .sum()
)

print("DOE borough-level (latest year):")
print(doe_boro)


Latest DOE year: 2019-20
DOE borough-level (latest year):
         borough  total_enrollment
0          Bronx          235448.0
1       Brooklyn          342332.0
2      Manhattan          180636.0
3         Queens          305623.0
4  Staten Island           67829.0


# Clean New Capacity Program Dataset

In [26]:
sca.columns = (
    sca.columns
       .str.strip()
       .str.lower()
       .str.replace(" ", "_")
       .str.replace("/", "_")
       .str.replace("#", "")
)

print("\nSCA cleaned columns:", sca.columns.tolist())


SCA cleaned columns: ['district', 'borough', 'small_ps__bldgs', 'small_ps__seats', 'small_ps_cost', 'ps_is__bldgs', 'ps_is__seats', 'ps_is_cost', 'is_hs__bldgs', 'is_hs__seats', 'is_hs_cost']


In [28]:
# clean borough names
sca["borough"] = sca["borough"].astype(str).str.title().str.strip()

# automatically detect all 'seats' and 'cost' columns
seat_cols = [c for c in sca.columns if "seats" in c.lower()]
cost_cols = [c for c in sca.columns if "cost" in c.lower()]

print("Seat columns detected:", seat_cols)
print("Cost columns detected:", cost_cols)

# convert those seat/cost columns to numeric (remove commas)
for col in seat_cols + cost_cols:
    sca[col] = (
        sca[col]
          .astype(str)
          .str.replace(",", "", regex=False)
          .replace({"": pd.NA})
    )
    sca[col] = pd.to_numeric(sca[col], errors="coerce")

# total seats and total cost per row (district level)
sca["total_new_capacity"] = sca[seat_cols].fillna(0).sum(axis=1)
sca["total_cost"] = sca[cost_cols].fillna(0).sum(axis=1)

# aggregate to borough level
sca_boro = (
    sca.groupby("borough", as_index=False)[["total_new_capacity", "total_cost"]]
       .sum()
)

print("SCA borough-level totals:")
print(sca_boro)

Seat columns detected: ['small_ps__seats', 'ps_is__seats', 'is_hs__seats']
Cost columns detected: ['small_ps_cost', 'ps_is_cost', 'is_hs_cost', 'total_cost']
SCA borough-level totals:
         borough  total_new_capacity  total_cost
0          Bronx                5208      998.32
1       Brooklyn               14718     2513.12
2      Manhattan                4087      687.44
3         Queens               18533     3748.94
4  Staten Island                2082      501.28


# Merge Data

In [31]:
merged = doe_boro.merge(sca_boro, on="borough", how="inner")

# seats per 1,000 students
merged["seats_per_1000_students"] = (
    1000 * merged["total_new_capacity"] / merged["total_enrollment"]
)

# dollar cost per student
merged["cost_per_student"] = (
    merged["total_cost"] / merged["total_enrollment"]
)

merged = merged.sort_values("borough").reset_index(drop=True)

print("Merged borough table with rates:")
print(merged)

Merged borough table with rates:
         borough  total_enrollment  total_new_capacity  total_cost  \
0          Bronx          235448.0                5208      998.32   
1       Brooklyn          342332.0               14718     2513.12   
2      Manhattan          180636.0                4087      687.44   
3         Queens          305623.0               18533     3748.94   
4  Staten Island           67829.0                2082      501.28   

   seats_per_1000_students  cost_per_student  
0                22.119534          0.004240  
1                42.993351          0.007341  
2                22.625612          0.003806  
3                60.640070          0.012267  
4                30.694836          0.007390  
